# Phishing Email Sentiment Analysis
## Analyzing Language Patterns in Phishing vs. Safe Emails

This notebook uses Python and NLTK to analyze sentiment and language patterns in phishing versus safe emails. We'll examine how attackers use urgency, fear, and pressure in their messages.

## Setup: Install Required Libraries
Run this cell first to install necessary packages and download NLTK data.

In [ ]:
# Import required libraries
import pandas as pd
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('punkt_tab')

print("Setup complete!")

---
# Part 1: Load & Inspect Data

**Objective:** Load the CSV file and explore the dataset to understand its structure and contents.

In [ ]:
# Load the dataset
df = pd.read_csv('sample_email_dataset.csv')

print("Dataset loaded successfully!")
print(f"Total emails: {len(df)}")

### Display First 5 Rows

In [ ]:
# Display the first 5 rows
df.head()

### Count Phishing vs Safe Emails

In [ ]:
# Count the distribution of phishing vs safe emails
email_counts = df['label'].value_counts()
print("\nEmail Distribution:")
print(email_counts)
print(f"\nPhishing emails: {email_counts['phishing']}")
print(f"Safe emails: {email_counts['safe']}")

# Visualize the distribution
plt.figure(figsize=(8, 5))
email_counts.plot(kind='bar', color=['#ff6b6b', '#51cf66'])
plt.title('Distribution of Phishing vs Safe Emails', fontsize=14, fontweight='bold')
plt.xlabel('Email Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Display Example Emails

In [ ]:
# Print 3 example phishing emails
print("=" * 80)
print("3 EXAMPLE PHISHING EMAILS")
print("=" * 80)

phishing_emails = df[df['label'] == 'phishing']['text'].head(3)
for idx, email in enumerate(phishing_emails, 1):
    print(f"\nPhishing Email {idx}:")
    print(f"{email}")
    print("-" * 80)

In [ ]:
# Print 3 example safe emails
print("=" * 80)
print("3 EXAMPLE SAFE EMAILS")
print("=" * 80)

safe_emails = df[df['label'] == 'safe']['text'].head(3)
for idx, email in enumerate(safe_emails, 1):
    print(f"\nSafe Email {idx}:")
    print(f"{email}")
    print("-" * 80)

### Part 1 Summary

**Dataset loaded:** 40 emails total (20 phishing, 20 safe) - perfectly balanced.

**Key observation:** Phishing emails use urgent/threatening language ("verify," "immediately," "suspended"), while safe emails are informational and professional ("attached," "scheduled," "reminder").

### Part 1 Explanation

**What we did:**
1. **Loaded the dataset** using pandas `read_csv()` function
2. **Displayed the first 5 rows** to understand the data structure (label and text columns)
3. **Counted phishing vs safe emails** using `value_counts()` to see the distribution
4. **Printed examples** to observe the different language patterns:
   - **Phishing emails** use urgent language, threats, and calls to immediate action
   - **Safe emails** have neutral, informative tone without pressure tactics

**Key Observations:**
- Phishing emails often contain words like "urgent," "immediately," "verify," "suspended," "confirm"
- Safe emails use professional, calm language for legitimate communications

---
# Part 2: Preprocessing with NLTK

**Objective:** Clean and prepare the text data for analysis by:
- Tokenization (splitting text into words)
- Lowercasing (normalizing text)
- Removing stopwords (common words like "the", "is", "a")
- Removing punctuation

In [ ]:
# Define preprocessing function
def preprocess_text(text):
    """
    Preprocess email text using NLTK:
    1. Tokenization - split text into individual words
    2. Lowercasing - convert all text to lowercase
    3. Remove stopwords - remove common words that don't add meaning
    4. Remove punctuation - remove special characters (including Unicode)
    
    Args:
        text (str): Raw email text
    
    Returns:
        list: Cleaned list of tokens
    """
    import unicodedata
    
    # Step 1: Tokenization - split text into words
    tokens = word_tokenize(text)
    
    # Step 2: Lowercasing - convert to lowercase
    tokens = [token.lower() for token in tokens]
    
    # Step 3 & 4: Remove stopwords and all punctuation (ASCII + Unicode)
    stop_words = set(stopwords.words('english'))
    tokens = [
        token for token in tokens 
        if token not in stop_words 
        and not all(unicodedata.category(char).startswith('P') for char in token)
        and token.strip()
    ]
    
    return tokens

print("Preprocessing function defined!")

### Apply Preprocessing to Dataset

In [ ]:
# Apply preprocessing to all emails
df['cleaned_tokens'] = df['text'].apply(preprocess_text)

print("Preprocessing complete!")
print(f"Total emails processed: {len(df)}")

### Show Original vs Cleaned Examples

In [ ]:
# Display original vs cleaned text for 2 phishing and 2 safe emails
print("=" * 80)
print("PHISHING EMAIL PREPROCESSING EXAMPLES")
print("=" * 80)

for idx in df[df['label'] == 'phishing'].head(2).index:
    print(f"\nExample {idx + 1}:")
    print(f"\nOriginal Email:")
    print(f"{df.loc[idx, 'text']}")
    print(f"\nCleaned Token List:")
    print(f"{df.loc[idx, 'cleaned_tokens']}")
    print("-" * 80)

In [ ]:
print("=" * 80)
print("SAFE EMAIL PREPROCESSING EXAMPLES")
print("=" * 80)

for idx in df[df['label'] == 'safe'].head(2).index:
    print(f"\nExample {idx + 1}:")
    print(f"\nOriginal Email:")
    print(f"{df.loc[idx, 'text']}")
    print(f"\nCleaned Token List:")
    print(f"{df.loc[idx, 'cleaned_tokens']}")
    print("-" * 80)

### Part 2 Explanation

**Preprocessing Steps:**

1. **Tokenization**: Breaks text into individual words (tokens)
   - Example: "Your account is locked" → ['Your', 'account', 'is', 'locked']

2. **Lowercasing**: Converts all text to lowercase for consistency
   - Example: ['Your', 'account'] → ['your', 'account']

3. **Remove Stopwords**: Filters out common words that don't carry much meaning
   - Removed: 'the', 'is', 'a', 'to', 'your', 'in', etc.
   - Keeps: 'account', 'locked', 'verify', 'urgent'

4. **Remove Punctuation**: Strips special characters
   - Removed: '.', ',', '!', '?', ':', etc.

**Why this matters:**
- Preprocessing helps us focus on meaningful words
- After cleaning, we can better analyze key terms attackers use
- Notice how phishing emails still retain urgent/threatening words even after cleaning

---
# Part 3: Sentiment Analysis

**Objective:** Use NLTK's VADER (Valence Aware Dictionary and sEntiment Reasoner) to:
- Compute sentiment scores for each email
- Compare average sentiment between phishing and safe emails
- Understand how negative language is used in phishing attacks

### Initialize VADER Sentiment Analyzer

In [ ]:
# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

print("VADER Sentiment Analyzer initialized!")
print("\nVADER returns 4 sentiment scores:")
print("- neg: Negative sentiment (0 to 1)")
print("- neu: Neutral sentiment (0 to 1)")
print("- pos: Positive sentiment (0 to 1)")
print("- compound: Overall sentiment (-1 to +1, normalized weighted composite)")
print("\nWe'll use the 'compound' score for our analysis.")

### Compute Sentiment Scores

In [ ]:
# Function to get compound sentiment score
def get_sentiment_score(text):
    """
    Calculate VADER sentiment compound score for text.
    
    Args:
        text (str): Email text
    
    Returns:
        float: Compound sentiment score (-1 to +1)
               -1 = very negative, 0 = neutral, +1 = very positive
    """
    scores = sia.polarity_scores(text)
    return scores['compound']

# Apply sentiment analysis to all emails
df['sentiment_score'] = df['text'].apply(get_sentiment_score)

print("Sentiment analysis complete!")
print(f"\nSample sentiment scores:")
print(df[['label', 'text', 'sentiment_score']].head(10))

### Calculate Average Sentiment Scores

In [ ]:
# Calculate average sentiment score for phishing emails
avg_phishing_sentiment = df[df['label'] == 'phishing']['sentiment_score'].mean()

# Calculate average sentiment score for safe emails
avg_safe_sentiment = df[df['label'] == 'safe']['sentiment_score'].mean()

print("=" * 80)
print("SENTIMENT ANALYSIS RESULTS")
print("=" * 80)
print(f"\nAverage Sentiment Score for Phishing Emails: {avg_phishing_sentiment:.4f}")
print(f"Average Sentiment Score for Safe Emails: {avg_safe_sentiment:.4f}")
print(f"\nDifference: {abs(avg_phishing_sentiment - avg_safe_sentiment):.4f}")
print("\nInterpretation:")
if avg_phishing_sentiment < avg_safe_sentiment:
    print("✓ Phishing emails have MORE NEGATIVE sentiment than safe emails")
    print("✓ This reflects the use of fear, urgency, and threatening language")
else:
    print("✓ Phishing emails have MORE POSITIVE sentiment than safe emails")
    print("✓ This may reflect the use of enticing offers or fake rewards")

### Visualize Sentiment Distribution

In [ ]:
# Create visualizations
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Bar chart comparing average sentiment
sentiment_comparison = pd.DataFrame({
    'Email Type': ['Phishing', 'Safe'],
    'Average Sentiment': [avg_phishing_sentiment, avg_safe_sentiment]
})

axes[0].bar(sentiment_comparison['Email Type'], 
            sentiment_comparison['Average Sentiment'],
            color=['#ff6b6b', '#51cf66'])
axes[0].set_title('Average Sentiment Score: Phishing vs Safe', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Compound Sentiment Score', fontsize=12)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
axes[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(sentiment_comparison['Average Sentiment']):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# Plot 2: Histogram showing distribution
df[df['label'] == 'phishing']['sentiment_score'].hist(
    bins=15, alpha=0.6, color='#ff6b6b', label='Phishing', ax=axes[1]
)
df[df['label'] == 'safe']['sentiment_score'].hist(
    bins=15, alpha=0.6, color='#51cf66', label='Safe', ax=axes[1]
)
axes[1].set_title('Sentiment Score Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Compound Sentiment Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Detailed Sentiment Breakdown

In [ ]:
# Get detailed sentiment statistics
print("=" * 80)
print("DETAILED SENTIMENT STATISTICS")
print("=" * 80)

print("\nPHISHING EMAILS:")
print(df[df['label'] == 'phishing']['sentiment_score'].describe())

print("\n" + "=" * 80)
print("\nSAFE EMAILS:")
print(df[df['label'] == 'safe']['sentiment_score'].describe())

### Show Most Negative and Most Positive Emails

In [ ]:
# Find most negative email
most_negative_idx = df['sentiment_score'].idxmin()
print("=" * 80)
print("MOST NEGATIVE EMAIL")
print("=" * 80)
print(f"Label: {df.loc[most_negative_idx, 'label']}")
print(f"Sentiment Score: {df.loc[most_negative_idx, 'sentiment_score']:.4f}")
print(f"\nText: {df.loc[most_negative_idx, 'text']}")

In [ ]:
# Find most positive email
most_positive_idx = df['sentiment_score'].idxmax()
print("=" * 80)
print("MOST POSITIVE EMAIL")
print("=" * 80)
print(f"Label: {df.loc[most_positive_idx, 'label']}")
print(f"Sentiment Score: {df.loc[most_positive_idx, 'sentiment_score']:.4f}")
print(f"\nText: {df.loc[most_positive_idx, 'text']}")

### Part 3 Explanation

**What is VADER Sentiment Analysis?**

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon and rule-based sentiment analysis tool specifically designed for social media text. It works well for short texts like emails.

**How it works:**
- Analyzes each word for positive/negative/neutral sentiment
- Considers punctuation (!!!) and capitalization (URGENT)
- Returns a compound score from -1 (very negative) to +1 (very positive)

**Key Findings:**

1. **Phishing emails typically have more negative sentiment** due to:
   - Threatening language ("suspended", "locked", "failure")
   - Urgency and fear tactics ("immediately", "final notice")
   - Negative consequences ("account will be locked")

2. **Safe emails tend to be neutral or slightly positive** because:
   - Professional, informative tone
   - No pressure or threats
   - Helpful, service-oriented language

3. **Exception**: Some phishing emails may have positive sentiment if they use:
   - Fake rewards ("Congratulations! You won...")
   - False opportunities ("You are eligible...")

**Security Insight:**
Sentiment analysis can be a useful feature for phishing detection systems. However, it should be combined with other features (keywords, sender information, links) for robust detection.

---
## Bonus: Common Words Analysis

Let's identify the most common words in phishing vs safe emails to further understand language patterns.

In [ ]:
from collections import Counter

# Get all tokens from phishing emails
phishing_tokens = []
for tokens in df[df['label'] == 'phishing']['cleaned_tokens']:
    phishing_tokens.extend(tokens)

# Get all tokens from safe emails
safe_tokens = []
for tokens in df[df['label'] == 'safe']['cleaned_tokens']:
    safe_tokens.extend(tokens)

# Count most common words
phishing_common = Counter(phishing_tokens).most_common(15)
safe_common = Counter(safe_tokens).most_common(15)

print("=" * 80)
print("TOP 15 MOST COMMON WORDS IN PHISHING EMAILS")
print("=" * 80)
for word, count in phishing_common:
    print(f"{word:20} {count:3} {'█' * count}")

print("\n" + "=" * 80)
print("TOP 15 MOST COMMON WORDS IN SAFE EMAILS")
print("=" * 80)
for word, count in safe_common:
    print(f"{word:20} {count:3} {'█' * count}")

---
## Summary & Conclusions

### Key Takeaways:

1. **Language Patterns:**
   - Phishing emails use urgent, threatening language
   - Common phishing words: "account", "verify", "click", "immediately", "confirm"
   - Safe emails use informative, professional language

2. **Sentiment Analysis:**
   - Phishing emails generally have more negative sentiment
   - This reflects the use of fear and pressure tactics
   - VADER effectively captures these emotional tones

3. **Security Implications:**
   - Users should be suspicious of emails with:
     - Urgent deadlines or threats
     - Requests to verify/confirm account information
     - Negative consequences for not acting
   - Sentiment analysis can complement other phishing detection methods

4. **NLTK Tools Used:**
   - Tokenization for text splitting
   - Stopword removal for cleaning
   - VADER for sentiment analysis

### Next Steps:
- Could extend this analysis with more sophisticated NLP techniques
- Build a machine learning classifier using these features
- Analyze other linguistic features (e.g., urgency keywords, imperative verbs)

---
## Export Results to JSON

Save all analysis results to a JSON file for future reference and sharing.

In [ ]:
import json

# Collect all analysis results
results = {
    "dataset_info": {
        "total_emails": len(df),
        "phishing_count": int(email_counts['phishing']),
        "safe_count": int(email_counts['safe'])
    },
    "sentiment_analysis": {
        "avg_phishing_sentiment": float(avg_phishing_sentiment),
        "avg_safe_sentiment": float(avg_safe_sentiment),
        "difference": float(abs(avg_phishing_sentiment - avg_safe_sentiment)),
        "phishing_stats": {
            "min": float(df[df['label'] == 'phishing']['sentiment_score'].min()),
            "max": float(df[df['label'] == 'phishing']['sentiment_score'].max()),
            "mean": float(df[df['label'] == 'phishing']['sentiment_score'].mean()),
            "std": float(df[df['label'] == 'phishing']['sentiment_score'].std())
        },
        "safe_stats": {
            "min": float(df[df['label'] == 'safe']['sentiment_score'].min()),
            "max": float(df[df['label'] == 'safe']['sentiment_score'].max()),
            "mean": float(df[df['label'] == 'safe']['sentiment_score'].mean()),
            "std": float(df[df['label'] == 'safe']['sentiment_score'].std())
        }
    },
    "extreme_examples": {
        "most_negative": {
            "label": df.loc[most_negative_idx, 'label'],
            "score": float(df.loc[most_negative_idx, 'sentiment_score']),
            "text": df.loc[most_negative_idx, 'text']
        },
        "most_positive": {
            "label": df.loc[most_positive_idx, 'label'],
            "score": float(df.loc[most_positive_idx, 'sentiment_score']),
            "text": df.loc[most_positive_idx, 'text']
        }
    },
    "word_frequency": {
        "top_phishing_words": [[word, int(count)] for word, count in phishing_common[:10]],
        "top_safe_words": [[word, int(count)] for word, count in safe_common[:10]]
    }
}

# Save to JSON file
with open('analysis_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("=" * 80)
print("RESULTS EXPORTED")
print("=" * 80)
print("\n✓ Analysis results saved to: analysis_results.json")
print(f"\nFile contains:")
print(f"  - Dataset information ({results['dataset_info']['total_emails']} emails)")
print(f"  - Sentiment analysis scores")
print(f"  - Extreme examples (most negative/positive)")
print(f"  - Top 10 words for each category")
print("\nYou can now share these results without re-running the notebook!")